In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [16]:
import ee
import json
from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient().get_secret("gee")
credentials = ee.ServiceAccountCredentials(
    email=json.loads(secret)['client_email'],
    key_data=secret
)
ee.Initialize(credentials)
print("GEE initialized!")

KeyboardInterrupt: 

In [ ]:
import ee
import geemap
import numpy as np

# Define Punjab region of interest
punjab = ee.Geometry.Rectangle([73.8, 29.5, 76.5, 32.5])

# ── Sentinel-1 SAR (Rabi season: Dec 2021 – Apr 2022) ──
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(punjab)
      .filterDate('2021-12-01', '2022-04-30')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
      .select(['VV', 'VH'])
      .mean()
      .clip(punjab))

# ── Sentinel-2 optical (same season, cloud cover < 20%) ──
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(punjab)
      .filterDate('2021-12-01', '2022-04-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .select(['B2','B3','B4','B8'])  # Blue, Green, Red, NIR
      .mean()
      .clip(punjab))

# Quick check — print band names
print("SAR bands:", s1.bandNames().getInfo())
print("Optical bands:", s2.bandNames().getInfo())
print("✅ Data loaded successfully!")

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

def get_thumbnail(image, bands, vis_params, region):
    url = image.select(bands).getThumbURL({
        'min': vis_params['min'],
        'max': vis_params['max'],
        'palette': vis_params.get('palette', []),
        'region': region,
        'dimensions': 512,
        'format': 'png'
    })
    response = requests.get(url)
    return Image.open(BytesIO(response.content))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Punjab — SAR vs Optical (Rabi 2021-22)', fontsize=14)

# SAR VV
img_vv = get_thumbnail(s1, ['VV'], {'min': -25, 'max': 0, 'palette': ['black','white']}, punjab)
axes[0].imshow(img_vv)
axes[0].set_title('SAR — VV band')
axes[0].axis('off')

# SAR VH
img_vh = get_thumbnail(s1, ['VH'], {'min': -30, 'max': -5, 'palette': ['black','white']}, punjab)
axes[1].imshow(img_vh)
axes[1].set_title('SAR — VH band')
axes[1].axis('off')

# Optical false colour
img_opt = get_thumbnail(s2, ['B8','B4','B3'], {'min': 0, 'max': 3000}, punjab)
axes[2].imshow(img_opt)
axes[2].set_title('Optical — False colour (crops = bright red)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('punjab_sar_vs_optical.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Done!")


In [ ]:
# ── Speckle filtering using a focal mean (Lee-approximation in GEE) ──
def apply_speckle_filter(image):
    vv_filtered = image.select('VV').focal_mean(radius=3, kernelType='square', units='pixels')
    vh_filtered = image.select('VH').focal_mean(radius=3, kernelType='square', units='pixels')
    return image.addBands(vv_filtered.rename('VV_filtered')).addBands(vh_filtered.rename('VH_filtered'))

s1_filtered = apply_speckle_filter(s1)

# ── Visualise before vs after speckle filtering ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('SAR Speckle Filtering — Punjab', fontsize=14)

# Before
img_before = get_thumbnail(s1, ['VV'], {'min': -25, 'max': 0, 'palette': ['black','white']}, punjab)
axes[0].imshow(img_before)
axes[0].set_title('Before — raw VV (speckled)')
axes[0].axis('off')

# After
img_after = get_thumbnail(s1_filtered, ['VV_filtered'], {'min': -25, 'max': 0, 'palette': ['black','white']}, punjab)
axes[1].imshow(img_after)
axes[1].set_title('After — filtered VV (smooth)')
axes[1].axis('off')

# VH filtered
img_vh = get_thumbnail(s1_filtered, ['VH_filtered'], {'min': -30, 'max': -5, 'palette': ['black','white']}, punjab)
axes[2].imshow(img_vh)
axes[2].set_title('After — filtered VH')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('speckle_filter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Speckle filtering done!")

In [ ]:
# Smaller study area — Ludhiana district (~50x50km patch)
study_area = ee.Geometry.Rectangle([75.5, 30.6, 76.2, 31.1])

# Recompute SAR and optical for smaller area
s1_small = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(study_area)
      .filterDate('2021-12-01', '2022-04-30')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
      .select(['VV', 'VH'])
      .mean()
      .clip(study_area))

s2_small = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(study_area)
      .filterDate('2021-12-01', '2022-04-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .select(['B2','B3','B4','B8'])
      .mean()
      .clip(study_area))

# Apply speckle filter
s1_small_filtered = apply_speckle_filter(s1_small)

# Stack all 6 bands
combined_small = s1_small_filtered.select(['VV_filtered','VH_filtered']).addBands(
                 s2_small.select(['B2','B3','B4','B8']))

url = combined_small.getDownloadURL({
    'region': study_area,
    'scale': 100,
    'crs': 'EPSG:4326',
    'format': 'GEO_TIFF',
    'filePerBand': False
})

print("Downloading...")
response = requests.get(url, stream=True)
output_path = '/kaggle/working/ludhiana_rabi_2022.tif'

with open(output_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

size_mb = os.path.getsize(output_path) / (1024*1024)
print(f"✅ Saved — {size_mb:.1f} MB")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Open and inspect the file
with rasterio.open('/kaggle/working/ludhiana_rabi_2022.tif') as src:
    print("Shape:", src.shape)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    data = src.read()  # shape: (bands, height, width)

print("\nBand order: VV, VH, B2, B3, B4, B8")
print("Data shape:", data.shape)

# Quick stats per band
band_names = ['VV', 'VH', 'B2(Blue)', 'B3(Green)', 'B4(Red)', 'B8(NIR)']
for i, name in enumerate(band_names):
    print(f"{name}: min={data[i].min():.2f}, max={data[i].max():.2f}, mean={data[i].mean():.2f}")

# Visualise all 6 bands
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Ludhiana — all 6 bands', fontsize=14)

for i, (ax, name) in enumerate(zip(axes.flat, band_names)):
    ax.imshow(data[i], cmap='gray')
    ax.set_title(name)
    ax.axis('off')

plt.tight_layout()
plt.savefig('all_bands.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ File looks good!")

In [ ]:
# Get crop labels from ESRI Land Cover 2022
labels = (ee.ImageCollection('projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS')
          .filterDate('2022-01-01', '2022-12-31')
          .mosaic()
          .clip(study_area))

# ESRI classes: 5 = crops, everything else = non-crop
# Convert to binary: 1 = crop, 0 = non-crop
crop_mask = labels.eq(5).rename('crop')

# Download labels
label_url = crop_mask.getDownloadURL({
    'region': study_area,
    'scale': 100,
    'crs': 'EPSG:4326',
    'format': 'GEO_TIFF',
})

response = requests.get(label_url, stream=True)
label_path = '/kaggle/working/ludhiana_labels.tif'

with open(label_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

# Visualise labels vs NIR band
with rasterio.open(label_path) as src:
    labels_arr = src.read(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Labels — crop vs non-crop', fontsize=14)

axes[0].imshow(data[5], cmap='gray')
axes[0].set_title('B8 NIR (input)')
axes[0].axis('off')

axes[1].imshow(labels_arr, cmap='RdYlGn')
axes[1].set_title('Crop mask (green=crop, red=non-crop)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('crop_labels.png', dpi=150, bbox_inches='tight')
plt.show()

crop_pct = labels_arr.mean() * 100
print(f"✅ Labels saved! Crop coverage: {crop_pct:.1f}%")

In [ ]:
with rasterio.open('/kaggle/working/ludhiana_rabi_2022.tif') as src:
    input_shape = src.shape

with rasterio.open('/kaggle/working/ludhiana_labels.tif') as src:
    label_shape = src.shape

print(f"Input shape: {input_shape}")
print(f"Label shape: {label_shape}")

if input_shape == label_shape:
    print("✅ Shapes match — ready for training!")
else:
    print("⚠️ Shape mismatch — needs fixing before training")

In [ ]:
# ── WEEK 2: Classical Baseline — NDVI threshold ──
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import jaccard_score, f1_score, accuracy_score

# Load data
with rasterio.open('/kaggle/working/ludhiana_rabi_2022.tif') as src:
    data = src.read().astype(np.float32)

with rasterio.open('/kaggle/working/ludhiana_labels.tif') as src:
    labels = src.read(1).astype(np.uint8)

# Band order: 0=VV, 1=VH, 2=B2, 3=B3, 4=B4, 5=B8
nir  = data[5]  # B8
red  = data[4]  # B4

# Compute NDVI
ndvi = (nir - red) / (nir - red + 1e-10)

# Threshold: NDVI > 0.3 = crop
ndvi_pred = (ndvi > 0.3).astype(np.uint8)

# Metrics
iou   = jaccard_score(labels.flatten(), ndvi_pred.flatten())
f1    = f1_score(labels.flatten(), ndvi_pred.flatten())
acc   = accuracy_score(labels.flatten(), ndvi_pred.flatten())

print("── Classical Baseline: NDVI threshold ──")
print(f"IoU:      {iou:.4f}")
print(f"F1:       {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('NDVI Baseline vs Ground Truth', fontsize=14)

axes[0].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0].set_title('NDVI map')
axes[0].axis('off')

axes[1].imshow(ndvi_pred, cmap='RdYlGn')
axes[1].set_title(f'NDVI prediction (IoU={iou:.3f})')
axes[1].axis('off')

axes[2].imshow(labels, cmap='RdYlGn')
axes[2].set_title('Ground truth')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('ndvi_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scale Sentinel-2 bands to reflectance (0-1)
nir_scaled = data[5] / 10000.0
red_scaled = data[4] / 10000.0

# Recompute NDVI correctly
ndvi = (nir_scaled - red_scaled) / (nir_scaled + red_scaled + 1e-10)

print("NDVI stats after scaling:")
print(f"  min:    {ndvi.min():.4f}")
print(f"  max:    {ndvi.max():.4f}")
print(f"  mean:   {ndvi.mean():.4f}")
print(f"  median: {np.median(ndvi):.4f}")

# Sweep thresholds again
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for t in thresholds:
    pred = (ndvi > t).astype(np.uint8)
    iou = jaccard_score(labels.flatten(), pred.flatten())
    f1  = f1_score(labels.flatten(), pred.flatten())
    results.append((t, iou, f1))
    print(f"Threshold {t:.1f} → IoU: {iou:.4f}, F1: {f1:.4f}")

best = max(results, key=lambda x: x[1])
print(f"\n✅ Best threshold: {best[0]} → IoU: {best[1]:.4f}")

# Visualise
best_pred = (ndvi > best[0]).astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'NDVI Baseline (scaled) — threshold {best[0]}', fontsize=14)

axes[0].imshow(ndvi, cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_title('NDVI map (corrected)')
axes[0].axis('off')

axes[1].imshow(best_pred, cmap='RdYlGn')
axes[1].set_title(f'Prediction (IoU={best[1]:.3f})')
axes[1].axis('off')

axes[2].imshow(labels, cmap='RdYlGn')
axes[2].set_title('Ground truth')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('ndvi_baseline_corrected.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Classical Baseline: SAR backscatter threshold ──

# Band order: 0=VV, 1=VH
vv = data[0]  # already in dB, no scaling needed
vh = data[1]

# VH/VV ratio — crops have distinctive ratio vs urban/water
# Crops: VH around -17 to -12 dB, urban: VH very high due to corner reflection
vh_vv_ratio = vh - vv  # in dB, subtraction = division in linear scale

print("VH-VV ratio stats:")
print(f"  min:    {vh_vv_ratio.min():.4f}")
print(f"  max:    {vh_vv_ratio.max():.4f}")
print(f"  mean:   {vh_vv_ratio.mean():.4f}")
print(f"  median: {np.median(vh_vv_ratio):.4f}")

# Sweep thresholds
thresholds = [-10, -8, -7, -6, -5, -4, -3]
results_sar = []

for t in thresholds:
    # crop = ratio above threshold (crops have less negative ratio than water)
    pred = (vh_vv_ratio > t).astype(np.uint8)
    iou = jaccard_score(labels.flatten(), pred.flatten())
    f1  = f1_score(labels.flatten(), pred.flatten())
    results_sar.append((t, iou, f1))
    print(f"Threshold {t} → IoU: {iou:.4f}, F1: {f1:.4f}")

best_sar = max(results_sar, key=lambda x: x[1])
print(f"\n✅ Best SAR threshold: {best_sar[0]} → IoU: {best_sar[1]:.4f}")

# Visualise
best_sar_pred = (vh_vv_ratio > best_sar[0]).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'SAR Baseline — VH/VV ratio threshold {best_sar[0]} dB', fontsize=14)

axes[0].imshow(vh_vv_ratio, cmap='RdYlGn')
axes[0].set_title('VH-VV ratio map')
axes[0].axis('off')

axes[1].imshow(best_sar_pred, cmap='RdYlGn')
axes[1].set_title(f'SAR prediction (IoU={best_sar[1]:.3f})')
axes[1].axis('off')

axes[2].imshow(labels, cmap='RdYlGn')
axes[2].set_title('Ground truth')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('sar_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

**UNET on Optical**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ── Dataset ──
class CropDataset(Dataset):
    def __init__(self, images, masks, patch_size=64):
        self.patches = []
        self.masks = []
        h, w = masks.shape
        for i in range(0, h - patch_size, patch_size // 2):
            for j in range(0, w - patch_size, patch_size // 2):
                img_patch  = images[:, i:i+patch_size, j:j+patch_size]
                mask_patch = masks[i:i+patch_size, j:j+patch_size]
                self.patches.append(img_patch)
                self.masks.append(mask_patch)

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        return (torch.tensor(self.patches[idx], dtype=torch.float32),
                torch.tensor(self.masks[idx],   dtype=torch.long))

# ── Normalize ──
def normalize(data):
    out = np.zeros_like(data, dtype=np.float32)
    for i in range(data.shape[0]):
        mn, mx = data[i].min(), data[i].max()
        out[i] = (data[i] - mn) / (mx - mn + 1e-10)
    return out

data_norm = normalize(data)

# Optical only: B2, B3, B4, B8 (bands 2,3,4,5)
optical_bands = data_norm[2:6]

# Train/val split by patches
dataset = CropDataset(optical_bands, labels)
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

print(f"Total patches: {len(dataset)}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"Input shape per patch: {dataset[0][0].shape}")
print("✅ Dataset ready!")

In [ ]:
# ── U-Net Architecture ──
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=2):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(128, 256)
        self.up3   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3  = DoubleConv(256, 128)
        self.up2   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2  = DoubleConv(128, 64)
        self.up1   = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1  = DoubleConv(64, 32)
        self.final = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

# ── Training function ──
def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3):
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    print(f"Training on: {device}")
    best_iou = 0
    history  = {'train_loss': [], 'val_iou': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device)
                preds = model(imgs).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds.flatten())
                all_labels.extend(masks.numpy().flatten())

        val_iou = jaccard_score(all_labels, all_preds)
        history['train_loss'].append(train_loss / len(train_loader))
        history['val_iou'].append(val_iou)

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), '/kaggle/working/unet_optical.pth')

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Val IoU: {val_iou:.4f} | Best: {best_iou:.4f}")

    print(f"\n✅ Optical U-Net done! Best Val IoU: {best_iou:.4f}")
    return history, best_iou

# ── Train optical-only U-Net ──
model_optical = UNet(in_channels=4, out_channels=2)
history_optical, best_iou_optical = train_model(model_optical, train_loader, val_loader, epochs=30)

**Now let's train the SAR-only U-Net**

In [ ]:
# ── SAR only dataset ──
sar_bands = data_norm[0:2]  # VV, VH only

sar_dataset   = CropDataset(sar_bands, labels)
train_size    = int(0.8 * len(sar_dataset))
val_size      = len(sar_dataset) - train_size
sar_train_ds, sar_val_ds = torch.utils.data.random_split(
    sar_dataset, [train_size, val_size]
)

sar_train_loader = DataLoader(sar_train_ds, batch_size=16, shuffle=True)
sar_val_loader   = DataLoader(sar_val_ds,   batch_size=16, shuffle=False)

# ── Train SAR-only U-Net ──
model_sar = UNet(in_channels=2, out_channels=2)
history_sar, best_iou_sar = train_model(
    model_sar, sar_train_loader, sar_val_loader, epochs=30
)

**SAR + Optical Fusion**

In [ ]:
# ── Fusion dataset — all 6 bands ──
all_bands = data_norm[0:6]  # VV, VH, B2, B3, B4, B8

fusion_dataset = CropDataset(all_bands, labels)
train_size     = int(0.8 * len(fusion_dataset))
val_size       = len(fusion_dataset) - train_size
fusion_train_ds, fusion_val_ds = torch.utils.data.random_split(
    fusion_dataset, [train_size, val_size]
)

fusion_train_loader = DataLoader(fusion_train_ds, batch_size=16, shuffle=True)
fusion_val_loader   = DataLoader(fusion_val_ds,   batch_size=16, shuffle=False)

# ── Train fusion U-Net (6 input channels) ──
model_fusion = UNet(in_channels=6, out_channels=2)
history_fusion, best_iou_fusion = train_model(
    model_fusion, fusion_train_loader, fusion_val_loader, epochs=30
)

In [ ]:
# Retrain optical with correct save path
def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, save_path='model.pth'):
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    print(f"Training on: {device}")
    best_iou = 0
    history  = {'train_loss': [], 'val_iou': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device)
                preds = model(imgs).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds.flatten())
                all_labels.extend(masks.numpy().flatten())

        val_iou = jaccard_score(all_labels, all_preds)
        history['train_loss'].append(train_loss / len(train_loader))
        history['val_iou'].append(val_iou)

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), save_path)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Val IoU: {val_iou:.4f} | Best: {best_iou:.4f}")

    print(f"\n✅ Done! Best Val IoU: {best_iou:.4f}")
    return history, best_iou

# Retrain all three with correct save paths
model_optical = UNet(in_channels=4, out_channels=2)
history_optical, best_iou_optical = train_model(
    model_optical, train_loader, val_loader, 
    epochs=30, save_path='/kaggle/working/unet_optical.pth'
)

model_sar = UNet(in_channels=2, out_channels=2)
history_sar, best_iou_sar = train_model(
    model_sar, sar_train_loader, sar_val_loader,
    epochs=30, save_path='/kaggle/working/unet_sar.pth'
)

model_fusion = UNet(in_channels=6, out_channels=2)
history_fusion, best_iou_fusion = train_model(
    model_fusion, fusion_train_loader, fusion_val_loader,
    epochs=30, save_path='/kaggle/working/unet_fusion.pth'
)

In [ ]:
# ── Visual comparison of all 3 models ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def predict_full(model, bands, patch_size=64):
    model.eval()
    _, h, w  = bands.shape
    pred_map = np.zeros((h, w), dtype=np.float32)
    count    = np.zeros((h, w), dtype=np.float32)

    with torch.no_grad():
        for i in range(0, h - patch_size + 1, patch_size // 2):
            for j in range(0, w - patch_size + 1, patch_size // 2):
                patch = torch.tensor(
                    bands[:, i:i+patch_size, j:j+patch_size],
                    dtype=torch.float32
                ).unsqueeze(0).to(device)
                out  = torch.softmax(model(patch), dim=1)[0, 1].cpu().numpy()
                pred_map[i:i+patch_size, j:j+patch_size] += out
                count[i:i+patch_size, j:j+patch_size]    += 1

    pred_map /= (count + 1e-10)
    return (pred_map > 0.5).astype(np.uint8)

# Load best weights
model_optical = UNet(in_channels=4, out_channels=2).to(device)
model_optical.load_state_dict(torch.load('/kaggle/working/unet_optical.pth'))
pred_optical = predict_full(model_optical.to(device), data_norm[2:6])

model_sar = UNet(in_channels=2, out_channels=2).to(device)
model_sar.load_state_dict(torch.load('/kaggle/working/unet_sar.pth'))
pred_sar = predict_full(model_sar, data_norm[0:2])

model_fusion.load_state_dict(torch.load('/kaggle/working/unet_fusion.pth') 
                              if torch.cuda.is_available() else 
                              torch.load('/kaggle/working/unet_fusion.pth', map_location='cpu'))
pred_fusion = predict_full(model_fusion, data_norm[0:6])

# Plot
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Model comparison — Ludhiana Rabi 2022', fontsize=14)

for ax, pred, title in zip(axes,
    [pred_optical, pred_sar, pred_fusion, labels],
    [f'Optical only (IoU=0.9637)',
     f'SAR only (IoU=0.9290)',
     f'Fusion (IoU=0.9646)',
     'Ground truth']):
    ax.imshow(pred, cmap='RdYlGn')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison saved!")

**Lets apply our model on Monsoon Season now**

In [ ]:
import ee
import geemap
import numpy as np

punjab = ee.Geometry.Rectangle([73.8, 29.5, 76.5, 32.5])

# ── Sentinel-1 SAR (Kharif/Monsoon: June–October 2022) ──
s1_monsoon = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(punjab)
      .filterDate('2022-06-01', '2022-10-31')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
      .select(['VV', 'VH'])
      .mean()
      .clip(punjab))

# ── Sentinel-2 optical (monsoon — NO cloud filter, to simulate real conditions) ──
s2_monsoon = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(punjab)
      .filterDate('2022-06-01', '2022-10-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))  # loose filter — keep cloudy
      .select(['B2','B3','B4','B8'])
      .mean()
      .clip(punjab))

print("SAR bands:",     s1_monsoon.bandNames().getInfo())
print("Optical bands:", s2_monsoon.bandNames().getInfo())
print("✅ Monsoon data loaded!")

In [ ]:
# Monsoon SAR — same Ludhiana area, different dates
s1_monsoon_small = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(study_area)
      .filterDate('2022-06-01', '2022-10-31')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
      .select(['VV', 'VH'])
      .mean()
      .clip(study_area))

# Monsoon Optical — loose cloud filter to simulate real cloudy conditions
s2_monsoon_small = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(study_area)
      .filterDate('2022-06-01', '2022-10-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))  # ← loose on purpose
      .select(['B2','B3','B4','B8'])
      .mean()
      .clip(study_area))

# Speckle filter + stack
s1_monsoon_filtered = apply_speckle_filter(s1_monsoon_small)
combined_monsoon = s1_monsoon_filtered.select(['VV_filtered','VH_filtered']).addBands(
                   s2_monsoon_small.select(['B2','B3','B4','B8']))

# Download directly
url_monsoon = combined_monsoon.getDownloadURL({
    'region': study_area,
    'scale': 100,
    'crs': 'EPSG:4326',
    'format': 'GEO_TIFF',
    'filePerBand': False
})

print("Downloading monsoon data...")
response = requests.get(url_monsoon, stream=True)
output_path_monsoon = '/kaggle/working/ludhiana_monsoon_2022.tif'
with open(output_path_monsoon, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

size_mb = os.path.getsize(output_path_monsoon) / (1024*1024)
print(f"✅ Saved — {size_mb:.1f} MB")

In [ ]:
import rasterio
import numpy as np

def load_tif(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

def normalize(data, percentile=98):
    out = np.zeros_like(data)
    for i in range(data.shape[0]):
        band = data[i]
        lo, hi = np.nanpercentile(band, 2), np.nanpercentile(band, percentile)
        out[i] = np.clip((band - lo) / (hi - lo + 1e-8), 0, 1)
    return out

monsoon = load_tif('/kaggle/working/ludhiana_monsoon_2022.tif')
print("Raw shape:", monsoon.shape)  # should be (6, H, W)

# Split into SAR and optical
sar_monsoon     = normalize(monsoon[0:2])   # VV, VH
optical_monsoon = normalize(monsoon[2:6])   # B2, B3, B4, B8

print("SAR shape:", sar_monsoon.shape)
print("Optical shape:", optical_monsoon.shape)

In [ ]:
import torch
import torch.nn.functional as F

def predict_full(model, data_norm):
    model.eval()
    tensor = torch.tensor(data_norm).unsqueeze(0).to(device)  # (1, C, H, W)
    
    _, _, H, W = tensor.shape
    
    # Pad to nearest multiple of 16
    pad_h = (16 - H % 16) % 16
    pad_w = (16 - W % 16) % 16
    tensor = F.pad(tensor, (0, pad_w, 0, pad_h))  # (left, right, top, bottom)
    
    with torch.no_grad():
        out = model(tensor)
        pred = torch.argmax(out, dim=1).squeeze().cpu().numpy()
    
    # Crop back to original size
    pred = pred[:H, :W]
    return pred

# Now run inference
fusion_monsoon = np.concatenate([sar_monsoon, optical_monsoon], axis=0)

pred_optical_monsoon = predict_full(model_optical, optical_monsoon)
pred_sar_monsoon     = predict_full(model_sar,     sar_monsoon)
pred_fusion_monsoon  = predict_full(model_fusion,  fusion_monsoon)

print("✅ Inference done!")
print("Prediction shape:", pred_optical_monsoon.shape)  # should be (558, 780)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Rabi vs Monsoon — Model Comparison (Ludhiana)', fontsize=14)

# ── Row 1: Rabi ──
axes[0,0].imshow(pred_optical, cmap='RdYlGn', vmin=0, vmax=1)
axes[0,0].set_title('Rabi — Optical (IoU=0.9637)', fontsize=10)
axes[0,0].axis('off')

axes[0,1].imshow(pred_sar, cmap='RdYlGn', vmin=0, vmax=1)
axes[0,1].set_title('Rabi — SAR (IoU=0.9290)', fontsize=10)
axes[0,1].axis('off')

axes[0,2].imshow(pred_fusion, cmap='RdYlGn', vmin=0, vmax=1)
axes[0,2].set_title('Rabi — Fusion (IoU=0.9646)', fontsize=10)
axes[0,2].axis('off')

# ── Row 2: Monsoon ──
axes[1,0].imshow(pred_optical_monsoon, cmap='RdYlGn', vmin=0, vmax=1)
axes[1,0].set_title('Monsoon — Optical (cloudy)', fontsize=10)
axes[1,0].axis('off')

axes[1,1].imshow(pred_sar_monsoon, cmap='RdYlGn', vmin=0, vmax=1)
axes[1,1].set_title('Monsoon — SAR', fontsize=10)
axes[1,1].axis('off')

axes[1,2].imshow(pred_fusion_monsoon, cmap='RdYlGn', vmin=0, vmax=1)
axes[1,2].set_title('Monsoon — Fusion', fontsize=10)
axes[1,2].axis('off')

plt.tight_layout()
plt.savefig('rabi_vs_monsoon_comparison.png', dpi=150)
plt.show()

**Fetch Cloud Mask (SCL band) from GEE**

In [ ]:
# Get a single image with most cloud cover instead of .mode()
scl_single = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(study_area)
      .filterDate('2022-06-01', '2022-10-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
      .filter(ee.Filter.gt('CLOUDY_PIXEL_PERCENTAGE', 30))  # ← actually cloudy images
      .select(['SCL'])
      .mosaic()   # ← takes first available image per pixel
      .clip(study_area))

url_scl2 = scl_single.getDownloadURL({
    'region': study_area,
    'scale': 100,
    'crs': 'EPSG:4326',
    'format': 'GEO_TIFF',
})

print("Downloading...")
response = requests.get(url_scl2, stream=True)
scl_path = '/kaggle/working/ludhiana_scl_monsoon.tif'
with open(scl_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

# Reload and recheck
with rasterio.open(scl_path) as src:
    scl = src.read(1).astype(np.float32)

cloud_classes = [3, 8, 9, 10]
cloud_mask = np.isin(scl, cloud_classes)

print("Unique SCL values:", np.unique(scl))
print(f"Cloud coverage: {cloud_mask.mean()*100:.1f}%")

In [ ]:
# Resize cloud mask to match prediction shape (558, 780)
H, W = pred_optical_monsoon.shape
cloud_mask_resized = resize(cloud_mask, (H, W), order=0, preserve_range=True).astype(bool)

print("Cloud mask resized to:", cloud_mask_resized.shape)

In [ ]:
# Start with optical prediction
final_prediction = pred_optical_monsoon.copy()

# Where cloudy → replace with SAR prediction
final_prediction[cloud_mask_resized] = pred_sar_monsoon[cloud_mask_resized]

print("✅ Decision fusion done!")
print(f"Pixels using SAR:     {cloud_mask_resized.sum():,} ({cloud_mask_resized.mean()*100:.1f}%)")
print(f"Pixels using Optical: {(~cloud_mask_resized).sum():,} ({(~cloud_mask_resized).mean()*100:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Decision-Level Fusion — Monsoon 2022 (Ludhiana)', fontsize=13)

plots = [
    (pred_optical_monsoon,  'Optical (cloudy)',      'RdYlGn'),
    (pred_sar_monsoon,      'SAR (cloud-free)',      'RdYlGn'),
    (cloud_mask_resized,    'Cloud Mask',            'Blues'),
    (final_prediction,      'Decision Fusion ✅',    'RdYlGn'),
]

for ax, (data, title, cmap) in zip(axes, plots):
    ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

# Legend for cloud mask
patch_cloud = mpatches.Patch(color='steelblue', label='Cloudy → use SAR')
patch_clear = mpatches.Patch(color='lightblue', label='Clear → use Optical')
axes[2].legend(handles=[patch_cloud, patch_clear], loc='lower left', fontsize=8)

plt.tight_layout()
plt.savefig('decision_fusion_monsoon.png', dpi=150)
plt.show()



**Accuracy Assessment — SAR Crop Monitoring (Decision Fusion)Ludhiana, Monsoon 2022**



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    ConfusionMatrixDisplay
)
 
 
# ─────────────────────────────────────────────
# 1. FLATTEN ARRAYS FOR SKLEARN
# ─────────────────────────────────────────────
def flatten(arr):
    """Flatten 2D raster to 1D, ignoring NaN/nodata."""
    flat = arr.flatten()
    return flat[~np.isnan(flat)].astype(int)
 
y_true       = flatten(labels)
y_opt        = flatten(pred_optical)
y_sar        = flatten(pred_sar)
y_fusion     = flatten(pred_fusion)
 
class_names  = ['Non-Crop', 'Crop']

# ─────────────────────────────────────────────
# 2. METRICS FUNCTION
# ─────────────────────────────────────────────
def compute_metrics(y_true, y_pred, name="Model"):
    oa    = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    f1    = f1_score(y_true, y_pred, average='weighted')
    prec  = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec   = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    iou_per_class = []
    for c in [0, 1]:
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        iou_per_class.append(tp / (tp + fp + fn + 1e-10))
    miou = np.mean(iou_per_class)
 
    return {
        'name'       : name,
        'OA'         : oa,
        'Kappa'      : kappa,
        'F1'         : f1,
        'Precision'  : prec,
        'Recall'     : rec,
        'mIoU'       : miou,
        'IoU_NonCrop': iou_per_class[0],
        'IoU_Crop'   : iou_per_class[1],
        'cm'         : confusion_matrix(y_true, y_pred)
    }
 
results = [
    compute_metrics(y_true, y_opt,    "Optical Only"),
    compute_metrics(y_true, y_sar,    "SAR Only"),
    compute_metrics(y_true, y_fusion, "Decision Fusion"),
]
 
 
# ─────────────────────────────────────────────
# 3. PRINT SUMMARY TABLE
# ─────────────────────────────────────────────
print("=" * 70)
print(f"{'Model':<20} {'OA':>6} {'Kappa':>7} {'F1':>6} {'mIoU':>7} {'IoU-Crop':>10}")
print("-" * 70)
for r in results:
    print(f"{r['name']:<20} {r['OA']:>6.4f} {r['Kappa']:>7.4f} "
          f"{r['F1']:>6.4f} {r['mIoU']:>7.4f} {r['IoU_Crop']:>10.4f}")
print("=" * 70)
 
# Detailed per-class report for fusion
print("\n── Detailed Classification Report: Decision Fusion ──")
print(classification_report(y_true, y_fusion, target_names=class_names))

In [ ]:
# ─────────────────────────────────────────────
# 4. VISUALIZE — CONFUSION MATRICES + BAR CHART
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle('Accuracy Assessment — SAR Crop Monitoring\nLudhiana, Monsoon 2022',
             fontsize=14, fontweight='bold', y=1.01)
 
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
 
# ── Row 1: Confusion Matrices ──
for col, r in enumerate(results):
    ax = fig.add_subplot(gs[0, col])
    cm_norm = r['cm'].astype(float) / r['cm'].sum(axis=1, keepdims=True)
    disp = ConfusionMatrixDisplay(confusion_matrix=r['cm'],
                                  display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
    ax.set_title(f"{r['name']}\nOA={r['OA']:.4f}  κ={r['Kappa']:.4f}",
                 fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelrotation=15)
 
# ── Row 2: Grouped bar chart of all metrics ──
ax_bar = fig.add_subplot(gs[1, :])
metrics_keys   = ['OA', 'Kappa', 'F1', 'mIoU', 'IoU_Crop', 'IoU_NonCrop']
metrics_labels = ['Overall\nAccuracy', 'Kappa', 'F1\n(weighted)',
                  'mIoU', 'IoU\n(Crop)', 'IoU\n(Non-Crop)']
colors         = ['#4C72B0', '#DD8452', '#55A868']
 
x     = np.arange(len(metrics_keys))
width = 0.22
 
for i, r in enumerate(results):
    vals = [r[k] for k in metrics_keys]
    bars = ax_bar.bar(x + i * width, vals, width,
                      label=r['name'], color=colors[i], alpha=0.88,
                      edgecolor='white', linewidth=0.6)
    for bar, v in zip(bars, vals):
        ax_bar.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.004,
                    f'{v:.3f}', ha='center', va='bottom',
                    fontsize=7.5, fontweight='bold')
 
ax_bar.set_xticks(x + width)
ax_bar.set_xticklabels(metrics_labels, fontsize=10)
ax_bar.set_ylim(0, 1.08)
ax_bar.set_ylabel('Score', fontsize=11)
ax_bar.set_title('Model Comparison — All Metrics', fontsize=11, fontweight='bold')
ax_bar.legend(fontsize=10, loc='lower right')
ax_bar.axhline(1.0, color='gray', linestyle='--', linewidth=0.7)
ax_bar.grid(axis='y', alpha=0.3)
ax_bar.spines[['top', 'right']].set_visible(False)
 
plt.savefig('/kaggle/working/accuracy_assessment.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Saved → /kaggle/working/accuracy_assessment.png")

In [ ]:
# ─────────────────────────────────────────────
# 5. FUSION IMPROVEMENT SUMMARY
# ─────────────────────────────────────────────
print("\n── Fusion Improvement Over Individual Sources ──")
for metric in ['OA', 'Kappa', 'F1', 'mIoU', 'IoU_Crop']:
    opt_val     = results[0][metric]
    sar_val     = results[1][metric]
    fusion_val  = results[2][metric]
    gain_vs_opt = (fusion_val - opt_val) * 100
    gain_vs_sar = (fusion_val - sar_val) * 100
    print(f"  {metric:<12} Fusion={fusion_val:.4f} | "
          f"vs Optical: {gain_vs_opt:+.2f}%  | "
          f"vs SAR: {gain_vs_sar:+.2f}%")
 